# Face Gen

In [ ]:
import os

if not os.path.exists("/content/stylegan2-ada-pytorch"):
    !pip install ninja -q
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

In [ ]:
import os
import gdown

if not os.path.exists("/content/ffhq.pkl"):
    gdown.download(
        "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl",
        "ffhq.pkl", quiet=False
    )

In [ ]:
import sys
sys.path.insert(0, "/content/stylegan2-ada-pytorch")

import warnings
warnings.filterwarnings("ignore")

import torch
import pickle
import numpy as np
from PIL import Image
import IPython.display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

with open("/content/ffhq.pkl", "rb") as f:
    G = pickle.load(f)["G_ema"].to(device)

def generate(seed):
    z = torch.from_numpy(np.random.RandomState(seed).randn(1, G.z_dim)).to(device)
    w = G.mapping(z, None, truncation_psi=0.7)
    img = G.synthesis(w, noise_mode="const")
    img = (img.permute(0,2,3,1) * 127.5 + 128).clamp(0,255).to(torch.uint8)
    return Image.fromarray(img[0].cpu().numpy())

# Generate 6 faces
cols = 3
grid = Image.new("RGB", (cols * 256, 2 * 256))
for i in range(6):
    img = generate(i)
    grid.paste(img.resize((256,256)), ((i % cols) * 256, (i // cols) * 256))

display(grid)

In [ ]:
def generate_w(seed):
    z = torch.from_numpy(np.random.RandomState(seed).randn(1, G.z_dim)).to(device)
    return G.mapping(z, None, truncation_psi=0.7)

def generate_from_w(w):
    img = G.synthesis(w, noise_mode="const")
    img = (img.permute(0,2,3,1) * 127.5 + 128).clamp(0,255).to(torch.uint8)
    return Image.fromarray(img[0].cpu().numpy())

# Interpolate between seed 0 and seed 1 in 8 steps
w_a = generate_w(0)
w_b = generate_w(1)
steps = 8

grid = Image.new("RGB", (steps * 128, 128))
for i, t in enumerate(np.linspace(0, 1, steps)):
    w_interp = (1 - t) * w_a + t * w_b
    img = generate_from_w(w_interp).resize((128, 128))
    grid.paste(img, (i * 128, 0))

display(grid)
print("← face A                                    face B →")